In [2]:
import pandas as pd

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind

In [3]:
df = pd.read_csv("../data/raw/ab_visual_assets.csv")

df.head()

,user_id,image_type,gender,age_group,customer_type,clicks,converted,time_on_page,purchase_value
0,1,studio,male,adult,new,0,0,40.41,0.0
1,2,lifestyle,male,young,new,0,0,45.43,0.0
2,3,lifestyle,male,young,returning,0,0,23.85,0.0
3,4,lifestyle,female,adult,new,0,0,38.83,0.0
4,5,studio,male,adult,new,0,0,46.64,0.0


## Teste 1

In [4]:
click_summary = df.groupby("image_type").agg(
    clicks=("clicks", "sum"),
    users=("user_id", "count"),
    ctr=("clicks", "mean")
)

click_summary

,clicks,users,ctr
image_type,,,
lifestyle,545,4924,0.110682
studio,438,5076,0.086288


In [5]:
clicks = click_summary["clicks"].values
users = click_summary["users"].values

z_stat, p_value = proportions_ztest(
    count=clicks,
    nobs=users,
    alternative="two-sided"
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Z-statistic: 4.0963
P-value: 0.0000


### Imagens lifestyle tiveram CTR significativamente maior do que imagens de estúdio.

## Teste 2

In [6]:
conversion_summary = df.groupby("image_type").agg(
    conversions=("converted", "sum"),
    users=("user_id", "count"),
    conversion_rate=("converted", "mean")
)

conversion_summary

,conversions,users,conversion_rate
image_type,,,
lifestyle,241,4924,0.048944
studio,205,5076,0.040386


In [7]:
conversions = conversion_summary["conversions"].values
users = conversion_summary["users"].values

z_stat, p_value = proportions_ztest(
    count=conversions,
    nobs=users,
    alternative="two-sided"
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Z-statistic: 2.0726
P-value: 0.0382


### A diferença de conversão foi estatisticamente significativa, embora com evidência mais moderada do que a observada para CTR.

## Teste 3

In [8]:
studio_time = df[df["image_type"] == "studio"]["time_on_page"]
lifestyle_time = df[df["image_type"] == "lifestyle"]["time_on_page"]

In [9]:
t_stat, p_value = ttest_ind(
    lifestyle_time,
    studio_time,
    equal_var=False
)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

T-statistic: 31.7346
P-value: 0.0000


### Usuários expostos a imagens lifestyle permaneceram significativamente mais tempo na página do que usuários expostos a imagens de estúdio.

## Teste 4

In [10]:
studio_revenue = df[df["image_type"] == "studio"]["purchase_value"]
lifestyle_revenue = df[df["image_type"] == "lifestyle"]["purchase_value"]

In [11]:
t_stat, p_value = ttest_ind(
    lifestyle_revenue,
    studio_revenue,
    equal_var=False
)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

T-statistic: 2.5766
P-value: 0.0100


### Como a distribuição de receita possui muitos valores zero e tende a ser assimétrica, análises futuras poderiam considerar testes não paramétricos ou métodos de bootstrap.

## Teste Estatístico - Interpretação Inicial

A análise do teste A/B comparou o desempenho de imagens de produtos em estúdio e lifestyle em relação à taxa de clique, taxa de conversão, tempo na página e receita por usuário.

Para métricas binárias, como cliques e conversões, foi utilizado o teste Z para duas proporções. Para métricas contínuas, como tempo na página e valor de compra, aplicou-se o teste t de Welch.

Um valor-p (p-value) abaixo de 0,05 indica que a diferença observada é estatisticamente significativa ao nível de significância de 5%.

No geral, os resultados sustentam a hipótese de que imagens de lifestyle performam melhor do que imagens de estúdio nesse cenário simulado de teste A/B.

## Statistical Testing - Initial Interpretation

The A/B test analysis compared the performance of studio and lifestyle product images across click-through rate, conversion rate, time on page, and revenue per user.

For binary metrics such as clicks and conversions, a two-proportion Z-test was used. For continuous metrics such as time on page and purchase value, Welch's t-test was applied.

A p-value below 0.05 indicates that the observed difference is statistically significant at the 5% significance level.

Overall, the results support the hypothesis that lifestyle images perform better than studio images in this simulated A/B testing scenario.

In [4]:
test_results = pd.DataFrame({
    "metric": [
        "CTR",
        "Conversion Rate",
        "Time on Page",
        "Revenue per User"
    ],
    "test": [
        "Two-proportion Z-test",
        "Two-proportion Z-test",
        "Welch's t-test",
        "Welch's t-test"
    ],
    "statistic": [
        4.0963,
        2.0726,
        31.7346,
        2.5766
    ],
    "p_value": [
        0.0000,
        0.0382,
        0.0000,
        0.0100
    ]
})

test_results["significant"] = test_results["p_value"] < 0.05

test_results

,metric,test,statistic,p_value,significant
0,CTR,Two-proportion Z-test,4.0963,0.0000,True
1,Conversion Rate,Two-proportion Z-test,2.0726,0.0382,True
2,Time on Page,Welch's t-test,31.7346,0.0000,True
3,Revenue per User,Welch's t-test,2.5766,0.0100,True
